# Browser History Topic Overview

Extracted from `browser_history_domain_overview.ipynb` starting at original Cell 21.\n

In [ ]:
import pandas as pd
from pathlib import Path

from initialisation import load_history_df

csv_path = Path("BrowserHistory.csv")
df = load_history_df(csv_path)

print(f"Loaded {len(df):,} rows for topic analysis from {csv_path.name}")

## 12) Expanded Topic Classification – Phase 1
Classifies **all** visited URLs (not just Google searches) using a priority-based approach:
1. **Domain mapping** (`DOMAIN_CATEGORY_MAP`) — fastest, most reliable
2. **Page title keywords** — contextual clues
3. **Domain name keywords** — fallback for unmapped domains
4. **URL path keywords** — last resort

Bilingual keywords (English + German). First match wins.\n

In [ ]:
# ---------------------------------------------------------------------------
# Phase 1 – Domain → Category direct mapping (100+ common domains)
# ---------------------------------------------------------------------------
DOMAIN_CATEGORY_MAP = {
    # Programming
    "github.com": "Programming",
    "stackoverflow.com": "Programming",
    "gitlab.com": "Programming",
    "bitbucket.org": "Programming",
    "python.org": "Python",
    "pypi.org": "Python",
    "nodejs.org": "JavaScript/Web",
    "npmjs.com": "JavaScript/Web",
    "reactjs.org": "JavaScript/Web",
    "vuejs.org": "JavaScript/Web",
    "rust-lang.org": "Systems/Backend",
    "go.dev": "Systems/Backend",
    "docs.microsoft.com": "Programming",
    "learn.microsoft.com": "Programming",
    "developer.mozilla.org": "JavaScript/Web",
    "w3schools.com": "JavaScript/Web",
    "hub.docker.com": "Systems/Backend",
    "docker.com": "Systems/Backend",
    # AI / ML
    "openai.com": "AI/LLMs",
    "chat.openai.com": "AI/LLMs",
    "claude.ai": "AI/LLMs",
    "huggingface.co": "ML/Deep Learning",
    "tensorflow.org": "ML/Deep Learning",
    "pytorch.org": "ML/Deep Learning",
    "kaggle.com": "Data Science",
    "deepseek.com": "AI/LLMs",
    "chat.deepseek.com": "AI/LLMs",
    "perplexity.ai": "AI/LLMs",
    "copilot.microsoft.com": "AI/LLMs",
    # Entertainment
    "youtube.com": "Entertainment/Media",
    "netflix.com": "Entertainment/Media",
    "twitch.tv": "Entertainment/Media",
    "imdb.com": "Entertainment/Media",
    "spotify.com": "Entertainment/Media",
    "reddit.com": "Entertainment/Media",
    "tiktok.com": "Entertainment/Media",
    "instagram.com": "Entertainment/Media",
    "facebook.com": "Entertainment/Media",
    "twitter.com": "Entertainment/Media",
    "x.com": "Entertainment/Media",
    "primevideo.com": "Entertainment/Media",
    "disneyplus.com": "Entertainment/Media",
    "store.steampowered.com": "Entertainment/Media",
    "steampowered.com": "Entertainment/Media",
    # Shopping
    "amazon.com": "Shopping/Ecommerce",
    "amazon.de": "Shopping/Ecommerce",
    "ebay.com": "Shopping/Ecommerce",
    "ebay.de": "Shopping/Ecommerce",
    "aliexpress.com": "Shopping/Ecommerce",
    "etsy.com": "Shopping/Ecommerce",
    "otto.de": "Shopping/Ecommerce",
    "zalando.de": "Shopping/Ecommerce",
    "mediamarkt.de": "Shopping/Ecommerce",
    "idealo.de": "Shopping/Ecommerce",
    "geizhals.de": "Shopping/Ecommerce",
    "conrad.de": "Shopping/Ecommerce",
    # News
    "bbc.com": "News/Current",
    "cnn.com": "News/Current",
    "dw.com": "News/Current",
    "spiegel.de": "News/Current",
    "zeit.de": "News/Current",
    "heise.de": "News/Current",
    "golem.de": "News/Current",
    "ard.de": "News/Current",
    "tagesschau.de": "News/Current",
    "sueddeutsche.de": "News/Current",
    "faz.net": "News/Current",
    # Travel
    "booking.com": "Travel/Location",
    "airbnb.com": "Travel/Location",
    "expedia.com": "Travel/Location",
    "tripadvisor.com": "Travel/Location",
    "maps.google.com": "Travel/Location",
    "openstreetmap.org": "Travel/Location",
    "flightradar24.com": "Travel/Location",
    # Finance
    "coinbase.com": "Finance/Investment",
    "crypto.com": "Finance/Investment",
    "investing.com": "Finance/Investment",
    "coingecko.com": "Finance/Investment",
    "binance.com": "Finance/Investment",
    "tradingview.com": "Finance/Investment",
    "finanzen.net": "Finance/Investment",
    # Work
    "linkedin.com": "Work/Career",
    "indeed.com": "Work/Career",
    "xing.com": "Work/Career",
    # Health
    "webmd.com": "Health/Medical",
    "mayoclinic.org": "Health/Medical",
    "healthline.com": "Health/Medical",
    "netdoktor.de": "Health/Medical",
    # Education
    "udemy.com": "Education/Learning",
    "coursera.org": "Education/Learning",
    "edx.org": "Education/Learning",
    "khanacademy.org": "Education/Learning",
    "codecademy.com": "Education/Learning",
    "wikipedia.org": "General Info",
    # Tools / Utilities
    "notion.so": "Tools/Utilities",
    "trello.com": "Tools/Utilities",
    "figma.com": "Tools/Utilities",
    "canva.com": "Tools/Utilities",
    "obsidian.md": "Tools/Utilities",
}

# ---------------------------------------------------------------------------
# Expanded topic keyword lists – 21 categories, bilingual (EN + DE)
# ---------------------------------------------------------------------------
TOPICS = {
    # Programming sub-categories
    "Python": [
        "python", "django", "flask", "pip", "pandas", "numpy", "jupyter",
        "anaconda", "virtualenv", "pycharm", "pytest", "poetry", "fastapi",
        "sqlalchemy", "python-programmierung", "pythonista",
    ],
    "JavaScript/Web": [
        "javascript", "react", "nodejs", "node.js", "typescript", "html",
        "css", "vue", "angular", "npm", "webpack", "vite", "svelte",
        "next.js", "nuxt", "tailwind", "bootstrap", "frontend",
        "web development", "webentwicklung",
    ],
    "Systems/Backend": [
        "rust", "golang", "c++", "c#", "java", "kotlin", "backend",
        "microservice", "postgresql", "mysql", "redis", "docker",
        "kubernetes", "devops", "linux", "systemd", "datenbank",
    ],
    "Mobile Dev": [
        "swift", "ios", "android", "flutter", "react native",
        "xcode", "android studio", "app development", "mobilentwicklung",
    ],
    # AI / ML sub-categories
    "AI/LLMs": [
        "copilot", "agent", "gpt", "llm", "chatgpt", "openai", "claude",
        "deepseek", "prompt", "gemini", "mistral", "ollama", "rag",
        "langchain", "llama", "mcp", "whisper", "ki", "chatbot",
        "sprachmodell", "kuenstliche intelligenz",
    ],
    "Data Science": [
        "data science", "data analysis", "visualization", "tableau",
        "power bi", "seaborn", "plotly", "notebook", "statistics",
        "regression", "datenanalyse", "datenwissenschaft",
    ],
    "ML/Deep Learning": [
        "neural", "deep learning", "tensorflow", "pytorch", "keras",
        "scikit", "sklearn", "reinforcement", "transformer", "fine-tuning",
        "embedding", "maschinelles lernen", "neuronales netz",
    ],
    # Hardware & Engineering
    "Hardware/Electronics": [
        "arduino", "raspberry", "circuit", "usb", "microcontroller",
        "fnirsi", "multimeter", "instek", "breadboard", "sensor",
        "electronics", "pcb", "soldering", "oscilloscope", "embedded",
        "schematic", "platine", "loetkolben", "widerstand",
    ],
    "Automotive/Vehicles": [
        "mercedes", "volvo", "bmw", "audi", "volkswagen", "toyota",
        "automobile", "engine", "motor", "vehicle", "driving", "repair",
        "maintenance", "fuel", "tire", "tesla", "electric vehicle",
        "auto", "kfz", "fahrzeug", "reparatur", "wartung",
    ],
    # Lifestyle domains
    "Health/Medical": [
        "health", "medical", "disease", "symptom", "doctor", "treatment",
        "medication", "hospital", "therapy", "wellness", "exercise",
        "nutrition", "mental health", "anxiety", "depression", "fitness",
        "gesundheit", "arzt", "krankheit", "medikament", "behandlung",
        "ernaehrung",
    ],
    "Entertainment/Media": [
        "movie", "film", "show", "gaming", "stream",
        "actor", "imdb", "music", "song", "artist", "album", "concert",
        "anime", "manga", "podcast", "serie", "musik", "spiel",
    ],
    "Travel/Location": [
        "hotel", "flight", "airport", "booking", "vacation", "travel",
        "route", "directions", "tourism", "hostel", "resort", "ticket",
        "reise", "urlaub", "flug", "stadtplan",
    ],
    "Finance/Investment": [
        "stock", "crypto", "bitcoin", "finance", "investment", "bank",
        "loan", "mortgage", "insurance", "tax", "savings", "trading",
        "portfolio", "ethereum", "defi",
        "aktie", "finanzen", "investition", "steuer", "versicherung",
        "krypto", "ersparnisse",
    ],
    "Sports": [
        "league", "football", "basketball", "soccer", "tennis",
        "championship", "tournament", "bundesliga", "formula 1", "f1",
        "cycling", "marathon", "fussball", "spieler", "mannschaft", "liga",
    ],
    "Food/Cooking": [
        "recipe", "cook", "food", "restaurant", "ingredient", "cooking",
        "baking", "cuisine", "dish", "menu", "delivery", "cafe", "vegan",
        "rezept", "kochen", "essen", "zutaten", "backen", "gericht",
    ],
    "Work/Career": [
        "resume", "interview", "salary", "employment", "career",
        "business", "company", "hiring", "recruit", "freelance", "startup",
        "bewerbung", "gehalt", "unternehmen", "karriere",
    ],
    "Education/Learning": [
        "tutorial", "learn", "class", "school", "university",
        "degree", "textbook", "exam", "study", "certification",
        "udemy", "coursera", "lecture",
        "kurs", "lernen", "schule", "studium", "pruefung", "zertifikat",
    ],
    # Broader categories (checked later in priority order)
    "Shopping/Ecommerce": [
        "amazon", "ebay", "shop", "buy", "price", "product", "sale",
        "aliexpress", "store", "deal", "coupon", "discount", "purchase",
        "checkout", "retail",
        "kaufen", "preis", "angebot", "rabatt", "einkaufen",
    ],
    "News/Current": [
        "news", "breaking", "update", "announced", "release", "launched",
        "latest", "announcement", "bulletin",
        "nachrichten", "aktuell", "meldung", "bericht",
    ],
    "Tools/Utilities": [
        "tool", "utility", "software", "application", "extension",
        "download", "installer", "plugin", "addon", "script", "cli",
        "command line", "terminal", "editor", "ide",
        "werkzeug", "programm", "herunterladen",
    ],
    "General Info": [
        "what is", "how to", "why", "explain", "definition", "meaning",
        " vs ", "comparison", "guide", "understand",
        "was ist", "wie", "warum", "erklaerung", "bedeutung", "vergleich",
        "anleitung",
    ],
}

print(f"DOMAIN_CATEGORY_MAP: {len(DOMAIN_CATEGORY_MAP)} domains")
print(f"TOPICS: {len(TOPICS)} categories, "
      f"{sum(len(v) for v in TOPICS.values())} total keywords")

In [ ]:
def _match_keywords(text: str) -> str:
    """Return first matching TOPICS category, or 'Other'."""
    if not text:
        return "Other"
    text_lower = text.lower()
    for topic, keywords in TOPICS.items():
        for kw in keywords:
            if kw in text_lower:
                return topic
    return "Other"


def classify_visit(url: str, domain: str, page_title: str) -> str:
    """Classify a browser visit using priority-based matching.

    Priority:
    1. Domain mapping (DOMAIN_CATEGORY_MAP)
    2. Page title keywords
    3. Domain name keywords
    4. URL path keywords
    """
    url = url if isinstance(url, str) else ""
    domain = domain if isinstance(domain, str) else ""
    page_title = page_title if isinstance(page_title, str) else ""

    # 1. Direct domain lookup
    if domain in DOMAIN_CATEGORY_MAP:
        return DOMAIN_CATEGORY_MAP[domain]

    # 2. Page title
    result = _match_keywords(page_title)
    if result != "Other":
        return result

    # 3. Domain name itself
    result = _match_keywords(domain)
    if result != "Other":
        return result

    # 4. Full URL (last resort)
    return _match_keywords(url)


# Apply to ALL rows in the DataFrame
page_title_col = "PageTitle" if "PageTitle" in df.columns else None

df["category"] = df.apply(
    lambda row: classify_visit(
        row["NavigatedToUrl"] if pd.notna(row["NavigatedToUrl"]) else "",
        row["domain"] if pd.notna(row["domain"]) else "",
        row[page_title_col] if (page_title_col and pd.notna(row[page_title_col])) else "",
    ),
    axis=1,
)

print(f"Classified {len(df):,} total visits")

In [ ]:
# Category distribution across ALL visits
category_counts = df["category"].value_counts()
total = len(df)

print(f"Category distribution – all {total:,} visits\n")
print(f"{'Category':<25} {'Count':>8}  {'%':>6}")
print("-" * 45)
for cat, count in category_counts.items():
    pct = count / total * 100
    print(f"{cat:<25} {count:>8,}  {pct:>5.1f}%")

other_pct = category_counts.get("Other", 0) / total * 100
# Threshold from Phase 1 plan: investigate clustering if > 40% falls into "Other"
OTHER_THRESHOLD_PCT = 40
print(f"\n→ 'Other' coverage: {other_pct:.1f}% ",
      "(Phase 2 clustering recommended)" if other_pct > OTHER_THRESHOLD_PCT else "(coverage acceptable)")

# Top 5 domains per non-Other category
print("\n\nTop 5 domains per category (excluding 'Other'):\n")
for cat in category_counts.index:
    if cat == "Other":
        continue
    top_domains = df[df["category"] == cat]["domain"].value_counts().head(5)
    domain_str = ", ".join(f"{d}({c})" for d, c in top_domains.items())
    print(f"{cat:<25} {domain_str}")

## 13) Phase 2 – Unsupervised Clustering of 'Other' Visits
Uses **TF-IDF char n-grams** + **KMeans** to discover natural groupings inside the
unclassified 'Other' bucket, helping identify topics that the keyword lists missed.\n

In [ ]:
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore")

other_df = df[df["category"] == "Other"].copy()
print(f"'Other' visits: {len(other_df):,} ({len(other_df) / len(df) * 100:.1f}% of total)")

# Minimum "Other" entries needed to make clustering statistically meaningful
MIN_CLUSTER_SIZE = 30
if len(other_df) < MIN_CLUSTER_SIZE:
    print("Not enough 'Other' entries to cluster meaningfully — skipping.")
else:
    page_title_col = "PageTitle" if "PageTitle" in other_df.columns else None
    title_series = (
        other_df[page_title_col].fillna("") if page_title_col else pd.Series("", index=other_df.index)
    )
    other_df["_cluster_text"] = other_df["domain"].fillna("") + " " + title_series

    # MAX_CLUSTERS caps granularity; MIN_CLUSTERS ensures meaningful groups;
    # CLUSTER_SIZE_RATIO: aim for ~50 visits per cluster
    MAX_CLUSTERS = 20
    MIN_CLUSTERS = 5
    CLUSTER_SIZE_RATIO = 50
    n_clusters = min(MAX_CLUSTERS, max(MIN_CLUSTERS, len(other_df) // CLUSTER_SIZE_RATIO))

    vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 3), max_features=500)
    X = vectorizer.fit_transform(other_df["_cluster_text"] )

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    other_df["cluster"] = kmeans.fit_predict(X)

    print(f"\nDiscovered {n_clusters} clusters in 'Other' visits.")
    print("Review these to find new keyword categories:\n")
    print(f"{'Cluster':>8}  {'Size':>6}  Top domains (count)")
    print("-" * 70)
    for cid in range(n_clusters):
        rows = other_df[other_df["cluster"] == cid]
        top = rows["domain"].value_counts().head(4)
        top_str = ', '.join(f"{d}({c})" for d, c in top.items())
        print(f"{cid:>8}  {len(rows):>6}  {top_str}")

    # Expose cluster labels on the main df for further inspection
    # Use boolean-mask positional assignment to avoid duplicate-index reindex errors
    other_mask = df["category"] == "Other"
    df.loc[other_mask, "other_cluster"] = other_df["cluster"].astype("Int64").to_numpy()
    print("\nCluster IDs written to df['other_cluster'] for further inspection.")

In [ ]:
df['other_cluster']